# Building a Retrieval-Augmented Generator (RAG) with LangChain, OpenAI and Pinecone

This notebook builds a question-answering system that answers from a **knowledge base** instead of from the model's memory alone. The knowledge base is six English Wikipedia articles about the topic of this lab: RAG, vector databases, embeddings, large language models, transformers and semantic search. They are stored as text files in `data/wikipedia/`.

The work is split into two parts that share almost all of their code:

| | Part 1 | Part 2 |
|---|---|---|
| Embeddings | OpenAI `text-embedding-3-small` | same |
| Chat model | OpenAI (via `langchain-openai`) | same |
| **Vector store** | `InMemoryVectorStore` (a Python list in this process) | **Pinecone** serverless index |
| Persistence | lost when the kernel stops | persists in the cloud |

The point of keeping everything else identical is that the comparison at the end isolates the one thing that changed: where the vectors live and how they are searched.

**Contents**

0. Setup
1. Architecture
2. Part 1: RAG with LangChain + OpenAI (in-memory vector store)
3. Part 2: the same RAG backed by Pinecone
4. Comparison between both implementations
5. Cleanup

## 0. Setup

Install the dependencies (see `README.md`) and create a `.env` file in the repository root with two keys:

```
OPENAI_API_KEY=sk-...
PINECONE_API_KEY=pcsk_...
```

The next cell loads them into the environment with `python-dotenv` and stops with a clear message if one is missing, so the notebook never fails half-way because of a missing key.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load OPENAI_API_KEY and PINECONE_API_KEY from the .env file in the repo root.
load_dotenv()

missing = [k for k in ("OPENAI_API_KEY", "PINECONE_API_KEY") if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}. Copy .env.example to .env and fill them in.")
print("API keys loaded.")

API keys loaded.


All tunable settings live in one cell. The chat model and the Pinecone index name can be overridden from `.env` without editing the notebook.

In [2]:
# Models
EMBEDDING_MODEL = "text-embedding-3-small"  # 1536-dimensional vectors
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1-mini")

# Chunking: ~1000 characters per chunk, with 200 characters shared between
# consecutive chunks so a sentence cut at a boundary still appears whole in one of them.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Number of chunks retrieved per question.
TOP_K = 4

# Pinecone
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "aygo-rag-wikipedia")
PINECONE_NAMESPACE = "wikipedia"
PINECONE_CLOUD, PINECONE_REGION = "aws", "us-east-1"  # the region available on Pinecone's free tier

DATA_DIR = Path("data/wikipedia")

## 1. Architecture

A RAG system has two pipelines. **Indexing** runs once, ahead of time, and turns documents into searchable vectors. **Querying** runs for every question: it finds the chunks closest in meaning to the question and gives them to the chat model as context.

```
INDEXING  (once, before any question)

  data/wikipedia/*.txt
          │
          ▼
  ┌──────────────┐   ┌──────────────────┐   ┌───────────────────┐   ┌──────────────────────────┐
  │    Loader    │──▶│  Text splitter   │──▶│  Embedding model  │──▶│       Vector store       │
  │ txt→Document │   │ 1000 chars, 200  │   │ OpenAI text-emb-  │   │ Part 1: InMemory (RAM)   │
  │ + metadata   │   │ overlap          │   │ 3-small → 1536-d  │   │ Part 2: Pinecone (cloud) │
  └──────────────┘   └──────────────────┘   └───────────────────┘   └──────────────────────────┘

QUERYING  (for every question)

  question ──▶ Embedding model ──▶ query vector
                                       │
                                       ▼
                         Vector store: top-k most similar chunks (cosine similarity)
                                       │
                                       ▼
               Prompt = system instructions + numbered chunks + question
                                       │
                                       ▼
                       OpenAI chat model ──▶ answer citing [1], [2], ...
```

How LangChain maps onto those boxes:

| Step | LangChain component |
|---|---|
| Loading | `Document` objects (`page_content` + `metadata`) |
| Splitting | `RecursiveCharacterTextSplitter` |
| Embedding | `OpenAIEmbeddings` |
| Storage and search | `InMemoryVectorStore` → `PineconeVectorStore` (same `VectorStore` interface) |
| Retrieval | `vector_store.as_retriever()` → a `Runnable` that maps a question to documents |
| Prompt | `ChatPromptTemplate` |
| Generation | `ChatOpenAI` + `StrOutputParser` |
| Wiring | LCEL (`|` operator) composes the steps into one chain |

The same flow as pseudocode:

```
index(documents):
    for doc in documents:
        for chunk in split(doc, size=1000, overlap=200):
            store.add(id(chunk), embed(chunk.text), chunk.text, chunk.metadata)

answer(question):
    q = embed(question)
    chunks = store.top_k(q, k=4, metric=cosine)
    prompt = SYSTEM + numbered(chunks) + question
    return llm(prompt), sources(chunks)
```

Because both vector stores implement LangChain's `VectorStore` interface, the querying chain is built by one function, `build_rag_chain(retriever)`, and Part 2 only passes it a different retriever.

## 2. Part 1: RAG with LangChain and OpenAI

### 2.1 Load the knowledge base

Each file in `data/wikipedia/` starts with a `Title:` line and a `Source:` line (written by `scripts/download_wikipedia.py`). The loader turns each file into a LangChain `Document` and moves those two lines into `metadata`, so every chunk later knows which article and URL it came from. That is what lets the answers cite their sources.

In [3]:
from langchain_core.documents import Document


def load_wikipedia_docs(data_dir: Path) -> list[Document]:
    docs = []
    for path in sorted(data_dir.glob("*.txt")):
        header, body = path.read_text(encoding="utf-8").split("\n\n", 1)
        meta = dict(line.split(": ", 1) for line in header.splitlines())
        docs.append(Document(page_content=body, metadata={"title": meta["Title"], "source": meta["Source"]}))
    return docs


docs = load_wikipedia_docs(DATA_DIR)
for d in docs:
    print(f"{d.metadata['title']:45s} {len(d.page_content):>8,d} chars")
print(f"\n{len(docs)} documents, {sum(len(d.page_content) for d in docs):,d} characters in total")

Large language model                            54,194 chars
Retrieval-augmented generation                  10,655 chars
Semantic search                                  2,209 chars
Transformer (deep learning)                    123,728 chars
Vector database                                  3,776 chars
Word embedding                                  10,186 chars

6 documents, 204,748 characters in total


### 2.2 Split into chunks

Whole articles are too long to be useful as retrieval units: one 120k-character article would fill the prompt and dilute the relevant part. `RecursiveCharacterTextSplitter` cuts on paragraph breaks first, then lines, then sentences, then words, so chunks tend to end at natural boundaries.

`add_start_index=True` records where each chunk begins in its article. Together with the source URL it gives every chunk a stable identity, which we turn into a deterministic ID. The same IDs are used in both vector stores, which (a) makes re-running the Pinecone upload idempotent (it overwrites instead of duplicating) and (b) lets the comparison check whether both stores return the *same* chunks.

In [4]:
import hashlib

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,  # stores metadata["start_index"]
)
chunks = splitter.split_documents(docs)

# Deterministic ID: hash of (source URL, position in the article).
chunk_ids = [
    hashlib.sha1(f"{c.metadata['source']}#{c.metadata['start_index']}".encode()).hexdigest()
    for c in chunks
]

print(f"{len(docs)} documents -> {len(chunks)} chunks")
print("\nExample chunk:", chunks[0].metadata)
print(chunks[0].page_content[:400], "...")

6 documents -> 308 chunks

Example chunk: {'title': 'Large language model', 'source': 'https://en.wikipedia.org/wiki/Large_language_model', 'start_index': 0}
A large language model (LLM) is an AI model (typically a neural network) trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts. They are the basis for many modern chatbots, such as ChatGPT, Claude, Gemini, Grok, Qwen and DeepSeek.
LLMs are typically based on the tr ...


### 2.3 Embeddings

An embedding model maps a text to a vector such that texts with similar *meaning* end up close together, even when they share no words. We ask the model for the vector of a short probe text to learn its dimension; Part 2 needs that number to create the Pinecone index.

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))
print(f"{EMBEDDING_MODEL} produces {EMBEDDING_DIM}-dimensional vectors")

text-embedding-3-small produces 1536-dimensional vectors


### 2.4 Index the chunks in an in-memory vector store

`InMemoryVectorStore` keeps every vector in a Python list and answers a search by computing the cosine similarity between the query vector and **all** stored vectors (exact, brute-force search). It needs no server and no account, which makes it the simplest baseline. Its limitations are what motivate Part 2: it disappears when the kernel stops, only this process can use it, and search time grows linearly with the number of vectors.

`add_documents` sends the chunks to OpenAI in batches to embed them, then stores the vectors. We time it for the comparison in section 4.

In [6]:
import time

from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore(embedding=embeddings)

t0 = time.perf_counter()
memory_store.add_documents(chunks, ids=chunk_ids)
memory_index_seconds = time.perf_counter() - t0

print(f"Indexed {len(chunks)} chunks in memory in {memory_index_seconds:.1f} s (includes the OpenAI embedding calls)")

Indexed 308 chunks in memory in 1.4 s (includes the OpenAI embedding calls)


### 2.5 Retrieval

Before involving the chat model, it is worth checking retrieval on its own. `similarity_search_with_score` embeds the question and returns the closest chunks together with their cosine similarity (1 = same direction, 0 = unrelated).

In [7]:
def show_hits(hits):
    for rank, (doc, score) in enumerate(hits, 1):
        preview = " ".join(doc.page_content.split())[:110]
        print(f"{rank}. [{score:.3f}] {doc.metadata['title']} (start {int(doc.metadata['start_index'])})\n   {preview}...")


question = "What problem does retrieval-augmented generation solve for LLMs?"
show_hits(memory_store.similarity_search_with_score(question, k=TOP_K))

1. [0.738] Large language model (start 14978)
   === Retrieval-augmented generation === Retrieval-augmented generation (RAG) is an approach that integrates LLM...
2. [0.663] Retrieval-augmented generation (start 0)
   Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and ...
3. [0.637] Retrieval-augmented generation (start 4843)
   Finally, the LLM can generate output based on both the query and the retrieved documents. Some models incorpor...
4. [0.636] Retrieval-augmented generation (start 3282)
   == Process == Retrieval-augmented generation (RAG) enhances large language models (LLMs) by incorporating an i...


### 2.6 Generation: the RAG chain

The chain has three stages, composed with LCEL:

1. **Retrieve**: the retriever turns the question into the top-k chunks. `RunnableParallel` keeps the original question next to them.
2. **Augment**: `format_docs` numbers the chunks (`[1] (Title) text...`) and the prompt template places them in the system message, with instructions to answer only from that context and to cite the numbers.
3. **Generate**: `ChatOpenAI` produces the answer and `StrOutputParser` extracts the text.

The chain returns both the answer and the retrieved documents, so the sources can be shown under each answer. Setting `temperature=0` makes the answers as repeatable as possible, which helps when comparing the two parts.

`build_rag_chain` takes the retriever as a parameter. That is the only seam Part 2 needs.

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant that answers questions using only the context below, "
     "which comes from Wikipedia articles.\n"
     "- Cite the passages you use with their number, e.g. [1] or [2][3].\n"
     "- If the context does not contain the answer, say that you don't know based on the "
     "knowledge base. Do not use outside knowledge.\n"
     "- Answer in at most a short paragraph.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(f"[{i}] ({d.metadata['title']}) {d.page_content}" for i, d in enumerate(docs, 1))


def build_rag_chain(retriever):
    generate = (
        (lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
        | prompt
        | llm
        | StrOutputParser()
    )
    # Output: {"question": ..., "docs": [...], "answer": "..."}
    return RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(answer=generate)


def ask(chain, question: str) -> dict:
    t0 = time.perf_counter()
    result = chain.invoke(question)
    elapsed = time.perf_counter() - t0
    print(f"Q: {question}\n\nA: {result['answer']}\n\nSources ({elapsed:.1f} s):")
    for i, d in enumerate(result["docs"], 1):
        print(f"  [{i}] {d.metadata['title']} (start {int(d.metadata['start_index'])}) {d.metadata['source']}")
    return result


memory_retriever = memory_store.as_retriever(search_kwargs={"k": TOP_K})
memory_rag = build_rag_chain(memory_retriever)

### 2.7 Testing the RAG

Three questions answered by different articles, and one that the knowledge base cannot answer. The last one checks that the model says so instead of answering from its own training data, which is the behaviour a RAG system should have.

In [9]:
_ = ask(memory_rag, "What problem does retrieval-augmented generation solve for LLMs?")

Q: What problem does retrieval-augmented generation solve for LLMs?

A: Retrieval-augmented generation (RAG) solves the problem of large language models (LLMs) being limited to their pre-existing training data by enabling them to retrieve and incorporate new, domain-specific, or updated information from external data sources. This means that when new information becomes available, the model does not need to be retrained; instead, the external knowledge base can be augmented with the updated information, allowing the LLM to generate responses based on both its training data and the retrieved documents. This addresses the challenge of keeping LLMs current and relevant without costly retraining [2][4].

Sources (1.9 s):
  [1] Large language model (start 14978) https://en.wikipedia.org/wiki/Large_language_model
  [2] Retrieval-augmented generation (start 0) https://en.wikipedia.org/wiki/Retrieval-augmented_generation
  [3] Retrieval-augmented generation (start 4843) https://en.wikipedia.or

In [10]:
_ = ask(memory_rag, "How does a vector database find similar items, and what is approximate nearest neighbor search?")

Q: How does a vector database find similar items, and what is approximate nearest neighbor search?

A: A vector database finds similar items by storing data as vector embeddings in a high-dimensional space, where each vector represents features of the data. Similarity search is performed by finding vectors close to a given input vector, meaning semantically similar data items have feature vectors near each other. To efficiently find these similar vectors, vector databases typically use approximate nearest neighbor (ANN) search algorithms, which quickly retrieve vectors that are close to the query vector without exhaustively comparing all vectors. ANN improves retrieval efficiency compared to exact methods like K-nearest neighbors (KNN) by approximating the nearest neighbors, enabling faster searches in large datasets [1][2][3][4].

Sources (1.6 s):
  [1] Vector database (start 0) https://en.wikipedia.org/wiki/Vector_database
  [2] Vector database (start 1464) https://en.wikipedia.org/w

In [11]:
_ = ask(memory_rag, "What is the attention mechanism in the transformer architecture?")

Q: What is the attention mechanism in the transformer architecture?

A: The attention mechanism in the transformer architecture is based on scaled dot-product attention units. For each unit, the model learns three weight matrices: the query weights (W^Q), the key weights (W^K), and the value weights (W^V). The module takes three sequences as input: a query sequence, a key sequence, and a value sequence. This mechanism allows the model to attend to relevant tokens for each token, capturing relationships and dependencies within the input data. Multiple attention heads, each with their own set of these weight matrices, enable the model to attend to different definitions of "relevance" in parallel, enhancing the model's ability to capture complex and long-range dependencies [1][4].

Sources (2.1 s):
  [1] Transformer (deep learning) (start 33167) https://en.wikipedia.org/wiki/Transformer_(deep_learning)
  [2] Transformer (deep learning) (start 0) https://en.wikipedia.org/wiki/Transformer_(

In [12]:
_ = ask(memory_rag, "Who won the 2022 FIFA World Cup?")

Q: Who won the 2022 FIFA World Cup?

A: The provided context does not contain information about the winner of the 2022 FIFA World Cup. Therefore, I don't know based on the knowledge base.

Sources (1.0 s):
  [1] Large language model (start 18998) https://en.wikipedia.org/wiki/Large_language_model
  [2] Large language model (start 37783) https://en.wikipedia.org/wiki/Large_language_model
  [3] Large language model (start 2929) https://en.wikipedia.org/wiki/Large_language_model
  [4] Transformer (deep learning) (start 119365) https://en.wikipedia.org/wiki/Transformer_(deep_learning)


### 2.8 Variant: retrieval as an agent tool

The LangChain RAG tutorial referenced by the lab (`docs.langchain.com/oss/python/deepagents/rag`) wires retrieval differently: instead of always retrieving once, it gives an **agent** a search tool and lets the model decide when and what to search. That allows several searches for a compound question, at the cost of extra model calls.

The tool below wraps any vector store, so the same agent can run on top of Pinecone in Part 2. It is shown here as a variant; the comparison in section 4 uses the chain from 2.6 because a fixed pipeline is easier to measure.

In [13]:
from langchain.agents import create_agent
from langchain_core.tools import tool


def make_search_tool(store):
    @tool
    def search_knowledge_base(query: str) -> str:
        '''Search the Wikipedia knowledge base (RAG, vector databases, embeddings, LLMs,
        transformers, semantic search) and return the most relevant passages.'''
        return format_docs(store.similarity_search(query, k=TOP_K))
    return search_knowledge_base


AGENT_PROMPT = (
    "Answer questions using the search_knowledge_base tool. You may call it more than once, "
    "e.g. once per sub-question. Base the answer only on what the tool returns, and name the "
    "article titles you used."
)


def run_agent(agent, question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for m in result["messages"]:
        for call in getattr(m, "tool_calls", None) or []:
            print(f"tool call -> search_knowledge_base({call['args']['query']!r})")
    print("\nA:", result["messages"][-1].content)


memory_agent = create_agent(llm, tools=[make_search_tool(memory_store)], system_prompt=AGENT_PROMPT)
run_agent(memory_agent, "Compare word embeddings and transformers: what does each one contribute to modern LLMs?")

tool call -> search_knowledge_base('word embeddings')
tool call -> search_knowledge_base('transformers in large language models')

A: Word embeddings and transformers each contribute distinct and complementary capabilities to modern large language models (LLMs):

Word Embeddings:
- Word embeddings represent words as real-valued vectors that encode semantic meaning, such that words closer in vector space are similar in meaning.
- They are generated using techniques like neural networks, dimensionality reduction, and probabilistic models.
- Word embeddings serve as the underlying input representation for NLP tasks, boosting performance in syntactic parsing, sentiment analysis, and more.
- Early popular methods include word2vec and fastText, which improved training speed and quality of vectors.
- More advanced embeddings like ELMo and BERT provide contextually meaningful, token-level embeddings that capture the multi-sense nature of words.

Transformers:
- Transformers are deep learning a

## 3. Part 2: Connecting the RAG to Pinecone

[Pinecone](https://www.pinecone.io/) is a managed vector database. Compared with the in-memory store it adds:

- **Persistence**: vectors are stored in the cloud and survive restarts; indexing is paid once, not on every run.
- **Sharing**: any process with the API key can query the same index (a web backend, a batch job, a teammate's notebook).
- **Scale**: it uses approximate nearest-neighbour (ANN) indexes instead of comparing against every vector, so search stays fast with millions of vectors.
- **Metadata filtering** evaluated inside the database, and **namespaces** to partition one index.

What changes in the architecture is only the vector-store box. LangChain's `PineconeVectorStore` implements the same `VectorStore` interface as `InMemoryVectorStore`, so `add_documents`, `similarity_search` and `as_retriever` keep working and the chain from Part 1 is reused unchanged.

```
                 ┌──────────────────── Pinecone (AWS us-east-1) ───────────────────┐
 chunks ─embed─▶ │ index "aygo-rag-wikipedia" · dim 1536 · cosine · serverless      │
                 │   namespace "wikipedia": {id, vector, metadata{text,title,…}}   │
 question ─embed─▶ query(vector, top_k=4) ──▶ ids + scores + metadata ─────────────┼─▶ prompt ─▶ LLM
                 └──────────────────────────────────────────────────────────────────┘
```

Note that embeddings are still computed by OpenAI on our side; Pinecone stores and searches the vectors. LangChain stores each chunk's text in the vector's metadata (under the key `text`), which is how a search result can be turned back into a `Document`.

### 3.1 Create (or reuse) the Pinecone index

An index is created with a fixed **dimension**, which must equal the embedding size (`EMBEDDING_DIM`, measured in 2.3), and a **metric**, cosine, the same one the in-memory store uses. `ServerlessSpec` means Pinecone manages capacity; we only choose cloud and region. `create_index` blocks until the index is ready. If the index already exists (e.g. from a previous run) it is reused.

In [14]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,  # must match the embedding model
        metric="cosine",          # same similarity measure as Part 1
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print(f"Created index {PINECONE_INDEX_NAME!r}")
else:
    print(f"Reusing existing index {PINECONE_INDEX_NAME!r}")

description = pc.describe_index(PINECONE_INDEX_NAME)
print(f"dimension={description.dimension}  metric={description.metric}  host={description.host}")
index = pc.Index(PINECONE_INDEX_NAME)

Reusing existing index 'aygo-rag-wikipedia'
dimension=1536  metric=cosine  host=aygo-rag-wikipedia-ed50dgw.svc.aped-4627-b74a.pinecone.io


### 3.2 Upload the chunks

**Change from Part 1:** `InMemoryVectorStore(...)` becomes `PineconeVectorStore(index=..., namespace=...)`. The call to `add_documents` is identical, with the same chunks and the same deterministic IDs. Because the IDs are deterministic, running this cell twice overwrites the same vectors (an *upsert*) instead of creating duplicates.

Pinecone is eventually consistent: an upserted vector can take a moment to become searchable. The loop after the upload waits until the index reports all vectors in the namespace, so the searches below see the complete data.

In [15]:
from langchain_pinecone import PineconeVectorStore

pinecone_store = PineconeVectorStore(index=index, embedding=embeddings, namespace=PINECONE_NAMESPACE)

t0 = time.perf_counter()
pinecone_store.add_documents(chunks, ids=chunk_ids)
pinecone_index_seconds = time.perf_counter() - t0
print(f"Upserted {len(chunks)} chunks in {pinecone_index_seconds:.1f} s (includes the OpenAI embedding calls)")


def namespace_count():
    ns = index.describe_index_stats().namespaces.get(PINECONE_NAMESPACE)
    return ns.vector_count if ns else 0


for _ in range(30):  # wait up to ~60 s for the writes to become visible
    if namespace_count() >= len(chunks):
        break
    time.sleep(2)
print(f"Vectors in namespace {PINECONE_NAMESPACE!r}: {namespace_count()}")

Upserted 308 chunks in 3.4 s (includes the OpenAI embedding calls)
Vectors in namespace 'wikipedia': 308


### 3.3 Retrieval from Pinecone

The same question as in 2.5. Each search now goes over the network: the question is embedded by OpenAI and the vector is sent to Pinecone, which returns the IDs, scores and metadata of the nearest vectors.

The chunks and their order match Part 1, and the scores agree to about three decimals. One detail: Pinecone stores every numeric metadata value as a float, so `start_index` comes back as e.g. `14978.0`; the display helpers cast it with `int()`.

In [16]:
show_hits(pinecone_store.similarity_search_with_score(question, k=TOP_K))

1. [0.737] Large language model (start 14978)
   === Retrieval-augmented generation === Retrieval-augmented generation (RAG) is an approach that integrates LLM...
2. [0.663] Retrieval-augmented generation (start 0)
   Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and ...
3. [0.637] Retrieval-augmented generation (start 4843)
   Finally, the LLM can generate output based on both the query and the retrieved documents. Some models incorpor...
4. [0.636] Retrieval-augmented generation (start 3282)
   == Process == Retrieval-augmented generation (RAG) enhances large language models (LLMs) by incorporating an i...


**Metadata filtering.** Pinecone can restrict a search to vectors whose metadata matches a filter, and it applies the filter inside the database during the search. Here the same kind of question is limited to the *Vector database* article. With the in-memory store this would mean scanning everything and filtering in Python.

In [17]:
show_hits(pinecone_store.similarity_search_with_score(
    "How are similar items found?",
    k=3,
    filter={"title": {"$eq": "Vector database"}},
))

1. [0.433] Vector database (start 1464)
   == Techniques == Common techniques for similarity search on high-dimensional vectors include: Hierarchical Nav...
2. [0.421] Vector database (start 1001)
   These feature vectors may be computed from the raw data using machine learning methods such as feature extract...
3. [0.272] Vector database (start 0)
   A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of ...


### 3.4 The RAG chain on top of Pinecone

**Change from Part 1:** a single line. `build_rag_chain` receives the Pinecone retriever instead of the in-memory one; the prompt, the model and the output format are the same objects as before.

In [18]:
pinecone_retriever = pinecone_store.as_retriever(search_kwargs={"k": TOP_K})
pinecone_rag = build_rag_chain(pinecone_retriever)

_ = ask(pinecone_rag, "What problem does retrieval-augmented generation solve for LLMs?")

Q: What problem does retrieval-augmented generation solve for LLMs?

A: Retrieval-augmented generation (RAG) solves the problem of large language models (LLMs) needing to access updated or domain-specific information that is not available in their original training data. Instead of retraining the entire model when new information becomes available, RAG allows LLMs to retrieve relevant documents from an external knowledge base and incorporate that information into their responses. This enables LLMs to generate answers based on both their pre-existing training and the most current or specialized data, improving accuracy and relevance without costly retraining [2][4].

Sources (1.9 s):
  [1] Large language model (start 14978) https://en.wikipedia.org/wiki/Large_language_model
  [2] Retrieval-augmented generation (start 0) https://en.wikipedia.org/wiki/Retrieval-augmented_generation
  [3] Retrieval-augmented generation (start 4843) https://en.wikipedia.org/wiki/Retrieval-augmented_generati

In [19]:
_ = ask(pinecone_rag, "How does a vector database find similar items, and what is approximate nearest neighbor search?")

Q: How does a vector database find similar items, and what is approximate nearest neighbor search?

A: A vector database finds similar items by storing data as vector embeddings in a high-dimensional space, where each vector represents features of the data. Similarity search is performed by comparing these vectors to find those that are semantically close to a given input vector. To efficiently find similar items, vector databases typically use approximate nearest neighbor (ANN) algorithms, which provide faster retrieval by approximating the closest vectors rather than computing exact nearest neighbors. This approach balances retrieval speed and accuracy, making it suitable for large-scale data. Techniques like Hierarchical Navigable Small World (HNSW) graphs are commonly used for ANN searches, enhancing performance in similarity search tasks [1][2].

Sources (1.7 s):
  [1] Vector database (start 0) https://en.wikipedia.org/wiki/Vector_database
  [2] Vector database (start 1464) https:

In [20]:
_ = ask(pinecone_rag, "Who won the 2022 FIFA World Cup?")

Q: Who won the 2022 FIFA World Cup?

A: The provided context does not contain information about the winner of the 2022 FIFA World Cup. Therefore, I don't know based on the knowledge base.

Sources (1.2 s):
  [1] Large language model (start 18998) https://en.wikipedia.org/wiki/Large_language_model
  [2] Large language model (start 37783) https://en.wikipedia.org/wiki/Large_language_model
  [3] Large language model (start 2929) https://en.wikipedia.org/wiki/Large_language_model
  [4] Transformer (deep learning) (start 119365) https://en.wikipedia.org/wiki/Transformer_(deep_learning)


The agent from 2.8 also runs on Pinecone by swapping its tool's store:

In [21]:
pinecone_agent = create_agent(llm, tools=[make_search_tool(pinecone_store)], system_prompt=AGENT_PROMPT)
run_agent(pinecone_agent, "Compare word embeddings and transformers: what does each one contribute to modern LLMs?")

tool call -> search_knowledge_base('word embeddings')
tool call -> search_knowledge_base('transformers in large language models')

A: Word embeddings and transformers each contribute distinct and complementary capabilities to modern large language models (LLMs):

Word Embeddings:
- Word embeddings represent words as real-valued vectors that encode semantic meaning, such that words closer in vector space are similar in meaning.
- They are generated using techniques like neural networks, dimensionality reduction, and probabilistic models.
- Word embeddings serve as the underlying input representation for NLP tasks, boosting performance in syntactic parsing, sentiment analysis, and more.
- Early popular methods include word2vec and fastText, which improved training speed and quality of vectors.
- More advanced embeddings like ELMo and BERT provide contextually meaningful, token-level embeddings that capture the multi-sense nature of words.

Transformers:
- Transformers are deep learning a

### 3.5 Persistence: reconnecting without re-indexing

This is the practical difference that matters most. The cell below builds a brand-new `PineconeVectorStore` from only the index name and namespace, as a separate application or a later session would. It does not load or embed any document, and it can answer immediately because the vectors are already stored in Pinecone. The in-memory store from Part 1 has no equivalent: after a restart it would have to re-embed every chunk.

In [22]:
fresh_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)
fresh_rag = build_rag_chain(fresh_store.as_retriever(search_kwargs={"k": TOP_K}))
_ = ask(fresh_rag, "What are some well-known word embedding models, and how do they differ?")

Q: What are some well-known word embedding models, and how do they differ?

A: Some well-known word embedding models include word2vec and fastText. Word2vec, created by a Google team led by Tomas Mikolov in 2013, is a toolkit that trains vector space models faster than previous approaches and helped popularize word embeddings for broader experimentation and practical application. FastText builds on this by representing words partly through character n-grams rather than treating each word as a single indivisible unit, allowing it to capture subword information. Additionally, more recent contextually-meaningful embeddings like ELMo and BERT differ from static embeddings by providing token-level embeddings, where each occurrence of a word has its own embedding that reflects its context, better capturing the multi-sense nature of words [3][4].

Sources (2.3 s):
  [1] Word embedding (start 0) https://en.wikipedia.org/wiki/Word_embedding
  [2] Word embedding (start 2656) https://en.wikipedia

## 4. Comparison between both implementations

Both stores hold the same chunks, embedded by the same model, and both use cosine similarity, so the comparison measures the store itself. Three things are measured:

1. **Indexing time**: the `add_documents` timings from 2.4 and 3.2. Both include the OpenAI embedding calls; Pinecone adds the network upload.
2. **Search latency**: each query is embedded **once** beforehand and then searched with `similarity_search_by_vector`, so the timing contains only the vector-store search (in-memory: a NumPy computation; Pinecone: an HTTPS round-trip). Each search is repeated several times and the median is reported.
3. **Retrieval agreement**: whether both stores return the same top-k chunk IDs for each query.

In [23]:
import statistics

import pandas as pd

eval_questions = [
    "What problem does retrieval-augmented generation solve for LLMs?",
    "How does a vector database find similar items?",
    "What is the attention mechanism in the transformer architecture?",
    "How is word2vec trained?",
    "What is the difference between semantic search and lexical search?",
    "What are hallucinations in large language models?",
]
REPEATS = 5


def timed_search(store, vector):
    times = []
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        hits = store.similarity_search_by_vector(vector, k=TOP_K)
        times.append((time.perf_counter() - t0) * 1000)
    return hits, statistics.median(times)


rows = []
for q in eval_questions:
    vector = embeddings.embed_query(q)  # embedded once, shared by both stores
    mem_hits, mem_ms = timed_search(memory_store, vector)
    pc_hits, pc_ms = timed_search(pinecone_store, vector)
    mem_ids = [d.id for d in mem_hits]
    pc_ids = [d.id for d in pc_hits]
    rows.append({
        "question": q,
        "in-memory ms": round(mem_ms, 2),
        "pinecone ms": round(pc_ms, 1),
        "same top-k (set)": f"{len(set(mem_ids) & set(pc_ids))}/{TOP_K}",
        "same order": mem_ids == pc_ids,
    })

latency = pd.DataFrame(rows)
latency

,question,in-memory ms,pinecone ms,same top-k (set),same order
0,What problem does retrieval-augmented generati...,9.76,108.3,4/4,True
1,How does a vector database find similar items?,10.07,136.6,4/4,True
2,What is the attention mechanism in the transfo...,10.56,105.5,4/4,True
3,How is word2vec trained?,10.46,112.8,4/4,True
4,What is the difference between semantic search...,9.43,104.5,4/4,True
5,What are hallucinations in large language models?,9.79,101.6,4/4,True


In [24]:
summary = pd.DataFrame({
    "metric": [
        "indexing time (s), incl. embeddings",
        "median search latency (ms)",
        "top-k agreement",
        "survives a kernel restart",
        "usable from other processes",
    ],
    "InMemoryVectorStore": [
        round(memory_index_seconds, 1),
        round(latency["in-memory ms"].median(), 2),
        "baseline",
        "no",
        "no",
    ],
    "Pinecone": [
        round(pinecone_index_seconds, 1),
        round(latency["pinecone ms"].median(), 1),
        f"{(latency['same top-k (set)'] == f'{TOP_K}/{TOP_K}').sum()}/{len(latency)} queries identical",
        "yes",
        "yes",
    ],
})
summary

,metric,InMemoryVectorStore,Pinecone
0,"indexing time (s), incl. embeddings",1.4,3.4
1,median search latency (ms),9.93,106.9
2,top-k agreement,baseline,6/6 queries identical
3,survives a kernel restart,no,yes
4,usable from other processes,no,yes


### 4.1 Answers side by side

The end-to-end RAG for one question with each store. Since retrieval agrees and `temperature=0`, the answers should be essentially the same; any difference comes from the order of the context chunks or from the model's own nondeterminism.

In [25]:
q = "What are hallucinations in large language models?"
mem_result = memory_rag.invoke(q)
pc_result = pinecone_rag.invoke(q)
print("IN-MEMORY:\n", mem_result["answer"], "\n")
print("PINECONE:\n", pc_result["answer"])

IN-MEMORY:
 Hallucinations in large language models refer to the generation of text that is syntactically fluent and appears factually sound but is actually internally inconsistent with the training data or factually incorrect. These hallucinations arise partly due to memorization of training data combined with extrapolation beyond factual boundaries. Models can sometimes output verbatim passages from training data when subjected to specific prompting sequences [1]. 

PINECONE:
 Hallucinations in large language models refer to the generation of text that is syntactically fluent and appears factually sound but is actually internally inconsistent with the training data or factually incorrect. These hallucinations occur partly due to the model memorizing training data and extrapolating beyond factual boundaries, sometimes outputting verbatim passages from the training data when prompted in specific ways [1].


### 4.2 Discussion

The numbers below come from the run saved in this notebook; they change slightly from run to run.

**Retrieval quality is identical.** Both stores returned the same 4 chunks in the same order for all 6 evaluation questions, and the end-to-end answers in 4.1 are the same. That is expected: both search the same OpenAI vectors with the same cosine metric. With 308 vectors, Pinecone's approximate (ANN) search finds the exact neighbours. Moving to Pinecone did not cost any answer quality.

**Latency favours the in-memory store at this size.** Searching in memory took about 10 ms, against about 110 ms for Pinecone. The in-memory search is a NumPy dot product over 308 vectors inside this process. A Pinecone search is an HTTPS round trip from Colombia to AWS `us-east-1`, so nearly all of its 110 ms is network time, not search time. Indexing shows the same pattern: Pinecone takes roughly 2–3× longer than the in-memory store (a few seconds in total), because it adds the upload to the shared embedding calls.

**Both numbers are small next to generation.** A full RAG answer took 1–3 s, almost all of it spent in the OpenAI chat call. The ~100 ms difference in retrieval is not noticeable to a user.

**Where Pinecone wins is everything that is not latency on a toy corpus:**

| | InMemoryVectorStore | Pinecone |
|---|---|---|
| Search algorithm | exact, compares against every vector: cost grows linearly with the corpus | ANN index: latency stays roughly flat as the corpus grows to millions of vectors |
| Capacity | limited by this process's RAM | managed storage, scales independently of the application |
| Persistence | lost on kernel restart; every run re-embeds (and pays for) all chunks | stored once; section 3.5 answered with zero re-embedding |
| Sharing | only this Python process | any client with the API key (API backend, workers, other notebooks) |
| Filtering | done in Python after loading everything | metadata filters and namespaces evaluated inside the database (3.3) |
| Operations | nothing to set up | account, API key, network dependency, eventual consistency after writes (3.2) |

**Conclusion.** For a notebook over a few hundred chunks, the in-memory store is simpler and faster. A RAG system that is deployed, has a large or growing knowledge base, or is used by more than one process needs a vector database. Thanks to LangChain's common `VectorStore` interface, the switch was a single constructor and one retriever argument, with no change to the prompt, the model or the chain.

## 5. Cleanup

The Pinecone index persists, and that is intended: section 3.5 depends on it, and the notebook reuses it on the next run. The free tier allows a limited number of indexes, so set `DELETE_INDEX = True` and run this cell when you are finished with the lab.

In [26]:
DELETE_INDEX = False

if DELETE_INDEX:
    pc.delete_index(PINECONE_INDEX_NAME)
    print(f"Deleted index {PINECONE_INDEX_NAME!r}")
else:
    print(f"Index {PINECONE_INDEX_NAME!r} kept ({namespace_count()} vectors in namespace {PINECONE_NAMESPACE!r}).")

Index 'aygo-rag-wikipedia' kept (308 vectors in namespace 'wikipedia').
